In [4]:
import numpy as np
from numba import njit

SIZE = 3
EMPTY = 0
P1 = 1
P2 = -1
max_moves = 100


def build_action_map(size):
    actions = []

    for r in range(size):
        for c in range(size):
            if not (r == 0 or r == size-1 or c == 0 or c == size-1):
                continue

            if c != 0:
                actions.append((r, c, r, 0))
            if c != size - 1:
                actions.append((r, c, r, size - 1))
            if r != 0:
                actions.append((r, c, 0, c))
            if r != size - 1:
                actions.append((r, c, size - 1, c))

    return np.array(actions, dtype=np.int8)  # (A, 4)

@njit
def get_valid_moves_numba(state, action_map):
    A = action_map.shape[0]
    valid = np.zeros(A, dtype=np.uint8)

    for i in range(A):
        r, c = action_map[i, 0], action_map[i, 1]
        piece = state[r, c]

        if piece == EMPTY or piece == P1:
            valid[i] = 1

    return valid

@njit
def push_numba(state, action):
    r, c, dr, dc = action
    size = state.shape[0]

    new = state.copy()

    if r == dr:
        # row shift
        if dc == 0:
            # push from right → shift right
            for j in range(c, 0, -1):
                new[r, j] = new[r, j-1]
            new[r, 0] = P1
        else:
            # push from left → shift left
            for j in range(c, size-1):
                new[r, j] = new[r, j+1]
            new[r, size-1] = P1

    else:
        # column shift
        if dr == 0:
            for i in range(r, 0, -1):
                new[i, c] = new[i-1, c]
            new[0, c] = P1
        else:
            for i in range(r, size-1):
                new[i, c] = new[i+1, c]
            new[size-1, c] = P1

    return new

@njit
def has_line_numba(state, player):
    size = state.shape[0]

    for i in range(size):
        if np.all(state[i, :] == player):
            return True
        if np.all(state[:, i] == player):
            return True

    diag1 = True
    diag2 = True

    for i in range(size):
        if state[i, i] != player:
            diag1 = False
        if state[i, size - 1 - i] != player:
            diag2 = False

    return diag1 or diag2

def encode_state_batch(states):
    # states: (B, S, S)

    p2 = (states == P2)
    empty = (states == EMPTY)
    p1 = (states == P1)

    encoded = np.stack((p2, empty, p1), axis=1).astype(np.float32)
    # (B, 3, S, S)

    return encoded

class QuixoFast:
    def __init__(self):
        self.size = SIZE
        self.action_map = build_action_map(SIZE)   # (A, 4)
        self.action_size = self.action_map.shape[0]

        # --- NEW: fast lookup ---
        self._action_to_index = {}
        for i in range(self.action_size):
            key = tuple(self.action_map[i])
            self._action_to_index[key] = i

    def get_initial_state(self):
        return np.zeros((self.size, self.size), dtype=np.int8)

    def change_perspective(self, state, player):
        return state * player

    def get_valid_moves(self, state):
        return get_valid_moves_numba(state, self.action_map)

    def get_next_state(self, state, action, player):
        canonical = state * player
        next_state = push_numba(canonical, self.action_map[action])
        return next_state * player

    def get_value_and_terminated(self, state, player, move_count):
        if has_line_numba(state, -player):
            return -1, True
        if has_line_numba(state, player):
            return 1, True
        if move_count >= max_moves:
            return 0, True
        return 0, False
    
    def encode_action(self, r, c, dr, dc):
        return self._action_to_index[(r, c, dr, dc)]

    def decode_action(self, action):
        return self.action_map[action]
    
    def get_encoded_state(self, state):
        """
        Encodes the state into a 3-channel representation for the ResNet.
        Channels: [Opponent's Pieces, Empty Squares, Current Player's Pieces]
        """
        # Constants used: P1 = 1, P2 = -1, EMPTY = 0
        if state.ndim == 2: # Single state (5, 5)
            return np.stack((state == -1, state == 0, state == 1)).astype(np.float32)
        
        # Batch of states (Batch, 5, 5)
        # We swap axes so the output is (Batch, 3, 5, 5)
        encoded = np.stack((state == -1, state == 0, state == 1)).astype(np.float32)
        return np.swapaxes(encoded, 0, 1)

    def get_opponent(self, player):
        """Returns the other player (-1 for 1, 1 for -1)."""
        return -player

    def get_opponent_value(self, value):
        """Flips the value for the other player's perspective."""
        return -value

In [3]:
import networkx as nx
import plotly.graph_objects as go

def generate_quixo_graph():
    game = QuixoFast()
    initial_state = game.get_initial_state()
    
    # Convert state to a hashable format for the graph
    def to_tuple(s): return tuple(s.flatten())

    G = nx.DiGraph()
    visited = set()
    # Queue stores (state_tuple, current_player)
    queue = [(to_tuple(initial_state), P1)]
    visited.add((to_tuple(initial_state), P1))

    print("Exploring state space...")
    while queue:
        curr_state_tuple, curr_player = queue.pop(0)
        curr_state = np.array(curr_state_tuple).reshape((SIZE, SIZE))
        
        # Check if game is already over
        _, terminated = game.get_value_and_terminated(curr_state, curr_player, 0)
        if terminated:
            continue

        # Get all valid moves
        valid_moves = game.get_valid_moves(curr_state)
        for action_idx, is_valid in enumerate(valid_moves):
            if is_valid:
                next_state = game.get_next_state(curr_state, action_idx, curr_player)
                next_player = game.get_opponent(curr_player)
                next_state_tuple = to_tuple(next_state)
                
                # Add edge: (current_state, player) -> (next_state, next_player)
                u = (curr_state_tuple, curr_player)
                v = (next_state_tuple, next_player)
                G.add_edge(u, v)

                if v not in visited:
                    visited.add(v)
                    queue.append(v)

    print(f"Graph generated with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

    # subgraph for speed
    G = G.subgraph(list(G.nodes)[:2000])
    # Calculate 3D Force-Directed Layout
    print("Relaxing graph (calculating 3D layout)...")
    pos = nx.spring_layout(G, dim=3, seed=216, k=20000, iterations=100)

    # Extract coordinates for Plotly
    edge_x, edge_y, edge_z = [], [], []
    for edge in G.edges():
        x0, y0, z0 = pos[edge[0]]
        x1, y1, z1 = pos[edge[1]]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])
        edge_z.extend([z0, z1, None])

    node_x = [pos[node][0] for node in G.nodes()]
    node_y = [pos[node][1] for node in G.nodes()]
    node_z = [pos[node][2] for node in G.nodes()]
    
    # Color nodes by player turn
    node_color = ['blue' if node[1] == P1 else 'red' for node in G.nodes()]
    node_text = [f"State: {node[0]}<br>Turn: {'P1' if node[1] == P1 else 'P2'}" for node in G.nodes()]

    # Create Plotly Figure
    fig = go.Figure(data=[
        go.Scatter3d(
            x=edge_x, y=edge_y, z=edge_z,
            line=dict(width=1, color='#888888'),
            hoverinfo='none',
            mode='lines'
        ),
        go.Scatter3d(
            x=node_x, y=node_y, z=node_z,
            mode='markers',
            marker=dict(size=4, color=node_color, opacity=0.8),
            text=node_text,
            hoverinfo='text'
        )
    ])

    fig.update_layout(
        title="Quixo 2x2 State Space (3D Relaxed Graph)",
        scene=dict(
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            zaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        ),
        margin=dict(l=0, r=0, b=0, t=40)
    )
    fig.show()

if __name__ == "__main__":
    generate_quixo_graph()

Exploring state space...
Graph generated with 22398 nodes and 152324 edges.
Relaxing graph (calculating 3D layout)...


ModuleNotFoundError: No module named 'scipy'

In [ ]:
import networkx as nx
import plotly.graph_objects as go
from collections import deque

def generate_quixo_graph(moves_depth=5):
    game = QuixoFast()
    initial_state = game.get_initial_state()
    
    def to_tuple(s): return tuple(s.flatten())

    G = nx.DiGraph()
    # Queue stores: (state_tuple, current_player, current_depth)
    start_node = (to_tuple(initial_state), P1)
    queue = deque([(start_node, 0)])
    visited = {start_node: 0}

    print(f"Exploring state space to depth {moves_depth}...")
    
    while queue:
        (curr_state_tuple, curr_player), depth = queue.popleft()
        
        if depth >= moves_depth:
            continue

        curr_state = np.array(curr_state_tuple).reshape((SIZE, SIZE))
        _, terminated = game.get_value_and_terminated(curr_state, curr_player, depth)
        
        if terminated:
            continue

        valid_moves = game.get_valid_moves(curr_state)
        for action_idx, is_valid in enumerate(valid_moves):
            if is_valid:
                next_state = game.get_next_state(curr_state, action_idx, curr_player)
                next_player = game.get_opponent(curr_player)
                next_state_tuple = to_tuple(next_state)
                
                u = (curr_state_tuple, curr_player)
                v = (next_state_tuple, next_player)
                
                G.add_edge(u, v)

                if v not in visited:
                    visited[v] = depth + 1
                    queue.append((v, depth + 1))

    print(f"Graph generated: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges.")

    # Relaxation / Layout
    print("Calculating 3D spring layout...")
    pos = nx.spring_layout(G, dim=3, seed=42, k=0.3)

    # Prepare Plotly Data
    edge_x, edge_y, edge_z = [], [], []
    for edge in G.edges():
        x0, y0, z0 = pos[edge[0]]
        x1, y1, z1 = pos[edge[1]]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])
        edge_z.extend([z0, z1, None])

    node_x = [pos[node][0] for node in G.nodes()]
    node_y = [pos[node][1] for node in G.nodes()]
    node_z = [pos[node][2] for node in G.nodes()]
    
    # Color by depth to visualize the progression
    node_depths = [visited[node] for node in G.nodes()]
    
    fig = go.Figure(data=[
        go.Scatter3d(
            x=edge_x, y=edge_y, z=edge_z,
            line=dict(width=1, color='#888'),
            hoverinfo='none',
            mode='lines'
        ),
        go.Scatter3d(
            x=node_x, y=node_y, z=node_z,
            mode='markers',
            marker=dict(
                size=4, 
                color=node_depths, 
                colorscale='Viridis', 
                colorbar=dict(title="Move Depth"),
                opacity=0.8
            ),
            text=[f"Depth: {visited[node]}<br>Turn: {'P1' if node[1] == P1 else 'P2'}" for node in G.nodes()],
            hoverinfo='text'
        )
    ])

    fig.update_layout(
        title=f"Quixo State Space (Depth {moves_depth})",
        scene=dict(xaxis_visible=False, yaxis_visible=False, zaxis_visible=False)
    )
    fig.show()

import numpy as np

def get_canonical_state(state_array):
    """
    Returns the lexicographically smallest version of the board 
    across all 8 symmetries (rotations and reflections).
    """
    symmetries = []
    curr = state_array
    for _ in range(4):
        curr = np.rot90(curr)
        symmetries.append(tuple(curr.flatten()))
        symmetries.append(tuple(np.fliplr(curr).flatten()))
    
    # Return the "representative" for this equivalence class
    return min(symmetries)

if __name__ == "__main__":
    # For SIZE=2, depth 5-8 covers most of the interesting space.
    # For SIZE=5, keep depth low (e.g., 3 or 4) to avoid long render times.
    generate_quixo_graph(moves_depth=3)

ModuleNotFoundError: No module named 'plotly'

In [11]:
import numpy as np
import networkx as nx
import plotly.graph_objects as go
from collections import deque

# --- Global / Constants (Ensure these match your QuixoFast class) ---
P1 = 1
SIZE = 3 # Change to 2 for small testing, 5 for full game

def get_canonical_state(state_array, size):
    """
    Returns the lexicographically smallest version of the board 
    across all 8 symmetries (rotations and reflections).
    """
    symmetries = []
    curr = state_array.reshape((size, size))
    for _ in range(4):
        curr = np.rot90(curr)
        symmetries.append(tuple(curr.flatten()))
        # Horizontal flip of the current rotation
        symmetries.append(tuple(np.fliplr(curr).flatten()))
    
    return min(symmetries)

def generate_quixo_graph(moves_depth=5):
    """
    Generates a NetworkX DiGraph by exploring the Quixo state space.
    If moves_depth is -1, it explores until the entire reachable space is found.
    """
    game = QuixoFast()
    initial_state = game.get_initial_state()
    size = game.size
    
    # Handle infinite depth
    limit = float('inf') if moves_depth == -1 else moves_depth

    G = nx.DiGraph()
    
    # Nodes are stored as: (canonical_flattened_tuple, current_player)
    start_canonical = get_canonical_state(initial_state, size)
    start_node = (start_canonical, P1)
    
    queue = deque([(start_node, 0)])
    visited = {start_node: 0}

    print(f"Generating graph (Depth Limit: {'Infinite' if moves_depth == -1 else moves_depth})...")
    
    while queue:
        curr_node, depth = queue.popleft()
        (curr_state_tuple, curr_player) = curr_node
        
        if depth >= limit:
            continue

        curr_state = np.array(curr_state_tuple).reshape((size, size))
        
        # Check if game is over
        _, terminated = game.get_value_and_terminated(curr_state, curr_player, depth)
        if terminated:
            continue

        valid_moves = game.get_valid_moves(curr_state)
        for action_idx, is_valid in enumerate(valid_moves):
            if is_valid:
                # 1. Get raw next state
                raw_next = game.get_next_state(curr_state, action_idx, curr_player)
                next_player = game.get_opponent(curr_player)
                
                # 2. Canonicalize it (The Symmetry Exploit)
                next_canonical = get_canonical_state(raw_next, size)
                v = (next_canonical, next_player)
                
                # 3. Add edge (u -> v)
                G.add_edge(curr_node, v)

                if v not in visited:
                    visited[v] = depth + 1
                    queue.append((v, depth + 1))

    print(f"Done! Graph has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")
    return G, visited

def plot_quixo_graph(G, visited, moves_depth):
    """
    Calculates a 3D spring layout and renders the graph using Plotly.
    """
    if G.number_of_nodes() == 0:
        print("Graph is empty. Increase depth or check game logic.")
        return

    print("Calculating 3D spring layout (this may take a moment)...")
    # k: optimal distance between nodes. iterations: smoothness of relaxation.
    for u, v, d in G.edges(data=True):
        G[u][v]['weight'] = 1.0 / (visited[u] + 1)
    pos = nx.spring_layout(G, dim=3, seed=42, k=0.8, iterations=500)

    # --- Edge Trace ---
    edge_x, edge_y, edge_z = [], [], []
    for edge in G.edges():
        x0, y0, z0 = pos[edge[0]]
        x1, y1, z1 = pos[edge[1]]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])
        edge_z.extend([z0, z1, None])

    edge_trace = go.Scatter3d(
        x=edge_x, y=edge_y, z=edge_z,
        line=dict(
            width=0.5,           # Very thin lines
            color='rgba(80,80,80,0.5)' # Low opacity (0.15) 
        ),
        hoverinfo='none',
        mode='lines'
    )

    # --- Node Trace ---
    node_x = [pos[node][0] for node in G.nodes()]
    node_y = [pos[node][1] for node in G.nodes()]
    node_z = [pos[node][2] for node in G.nodes()]
    node_depths = [visited[node] for node in G.nodes()]
    
    node_text = []
    for node in G.nodes():
        depth = visited[node]
        player = "P1" if node[1] == 1 else "P2"
        node_text.append(f"Depth: {depth}<br>Turn: {player}<br>State: {node[0]}")

    node_trace = go.Scatter3d(
        x=node_x, y=node_y, z=node_z,
        mode='markers',
        hoverinfo='text',
        text=node_text,
        marker=dict(
            showscale=True,
            colorscale='Viridis',
            color=node_depths,
            size=5,
            line_width=1,
            colorbar=dict(
                thickness=15,
                title=dict(text='Move Depth', side='right'),
                xanchor='left'
            )
        )
    )

    # --- Layout & Figure ---
    fig = go.Figure(data=[edge_trace, node_trace])
    
    depth_label = "Full Exploration" if moves_depth == -1 else f"Depth {moves_depth}"
    fig.update_layout(
        title=dict(
            text=f"Quixo Symmetrized State Space ({depth_label})",
            font=dict(size=18)
        ),
        paper_bgcolor='white',
        showlegend=False,
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False)
        ),
        margin=dict(l=0, r=0, b=0, t=50)
    )

    print("Opening 3D visualization...")
    fig.show()


ModuleNotFoundError: No module named 'plotly'

In [6]:
DEPTH = -1
SIZE = 2
graph, node_depth_map = generate_quixo_graph(moves_depth=DEPTH)


NameError: name 'generate_quixo_graph' is not defined

In [96]:

plot_quixo_graph(graph, node_depth_map, moves_depth=DEPTH)

Calculating 3D spring layout (this may take a moment)...
Opening 3D visualization...


In [68]:
# compute number of nodes and edges for quixo of size 2 to 5, full depth
for size in range(2, 6):
    SIZE = size
    game = QuixoFast()
    initial_state = game.get_initial_state()
    def to_tuple(s): return tuple(s.flatten())
    G = nx.DiGraph()
    queue = deque([(to_tuple(initial_state), P1)])
    visited = set(queue)
    while queue:
        curr_state_tuple, curr_player = queue.popleft()
        curr_state = np.array(curr_state_tuple).reshape((SIZE, SIZE))
        _, terminated = game.get_value_and_terminated(curr_state, curr_player, 0)
        if terminated:
            continue
        valid_moves = game.get_valid_moves(curr_state)
        for action_idx, is_valid in enumerate(valid_moves):
            if is_valid:
                next_state = game.get_next_state(curr_state, action_idx, curr_player)
                next_player = game.get_opponent(curr_player)
                next_state_tuple = to_tuple(next_state)
                u = (curr_state_tuple, curr_player)
                v = (next_state_tuple, next_player)
                G.add_edge(u, v)
                if v not in visited:
                    visited.add(v)
                    queue.append(v)
    print(f"SIZE={size}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")    


SIZE=2: 63 nodes, 180 edges
SIZE=3: 22398 nodes, 152324 edges


KeyboardInterrupt: 

In [100]:
import numpy as np
from collections import deque

def get_canonical_state(state_array, size):
    """Returns the lexicographical minimum of all 8 D4 symmetries."""
    symmetries = []
    curr = state_array.reshape((size, size))
    for _ in range(4):
        curr = np.rot90(curr)
        symmetries.append(tuple(curr.flatten()))
        symmetries.append(tuple(np.fliplr(curr).flatten()))
    return min(symmetries)

def log_quixo_unique_states(max_depth=5):
    game = QuixoFast() # Assumes your fast implementation is available
    size = game.size
    initial_state = game.get_initial_state()
    
    # We track (canonical_state_tuple, current_player)
    start_canonical = get_canonical_state(initial_state, size)
    start_node = (start_canonical, 1) # P1 starts
    
    # BFS structures
    visited = {start_node}
    current_layer = {start_node}
    
    print(f"{'Depth':<10} | {'Unique States (Symmetrized)':<25}")
    print("-" * 40)
    print(f"{0:<10} | {1:<25}")

    for depth in range(1, max_depth + 1):
        next_layer = set()
        
        for curr_node in current_layer:
            curr_state_tuple, curr_player = curr_node
            curr_state = np.array(curr_state_tuple).reshape((size, size))
            
            # Skip if game ended in previous move
            _, terminated = game.get_value_and_terminated(curr_state, curr_player, depth-1)
            if terminated:
                continue
                
            valid_moves = game.get_valid_moves(curr_state)
            next_player = game.get_opponent(curr_player)
            
            for action_idx, is_valid in enumerate(valid_moves):
                if is_valid:
                    raw_next = game.get_next_state(curr_state, action_idx, curr_player)
                    next_canonical = get_canonical_state(raw_next, size)
                    v = (next_canonical, next_player)
                    
                    if v not in visited:
                        visited.add(v)
                        next_layer.add(v)
        
        if not next_layer:
            print(f"Full state space explored at depth {depth-1}.")
            break
            
        print(f"{depth:<10} | {len(next_layer):<25}")
        current_layer = next_layer

if __name__ == "__main__":
    # 5x5 exploration is heavy; depth 4-5 is a good baseline to check growth
    SIZE = 4
    log_quixo_unique_states(max_depth=100)

Depth      | Unique States (Symmetrized)
----------------------------------------
0          | 1                        
1          | 2                        
2          | 19                       
3          | 124                      
4          | 867                      
5          | 4693                     
6          | 21370                    
7          | 71646                    
8          | 178985                   


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt
import imageio

def render_board(state):
    # Map values to colors: P2=-1 (black), EMPTY=0 (gray), P1=1 (white)
    color_map = {
        -1: "#ff0000",
         0: "#808080",
         1: "#0000ff"
    }
    img = np.vectorize(color_map.get)(state)

    fig, ax = plt.subplots()
    ax.imshow(img, vmin=0, vmax=1)

    # grid lines
    ax.set_xticks(np.arange(-.5, SIZE, 1))
    ax.set_yticks(np.arange(-.5, SIZE, 1))
    ax.grid(color='black', linestyle='-', linewidth=2)

    ax.set_xticklabels([])
    ax.set_yticklabels([])

    fig.canvas.draw()

    # convert to image array
    image = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8')
    image = image.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    plt.close(fig)

    return image


def play_random_game_and_make_gif(filename="quixo_random.gif"):
    game = QuixoFast()
    state = game.get_initial_state()

    player = P1
    move_count = 0

    frames = []
    frames.append(render_board(state))

    while True:
        valid = game.get_valid_moves(state)
        valid_indices = np.where(valid == 1)[0]

        if len(valid_indices) == 0:
            break

        action = np.random.choice(valid_indices)

        state = game.get_next_state(state, action, player)
        move_count += 1

        frames.append(render_board(state))

        value, done = game.get_value_and_terminated(state, player, move_count)
        if done:
            break

        player = game.get_opponent(player)

    # save gif
    imageio.mimsave(filename, frames, duration=1/30)

    print(f"Game finished in {move_count} moves. GIF saved to {filename}")

max_moves=100

# run it
play_random_game_and_make_gif()